In [1]:
import pandas as pd

corpus = pd.read_csv("../data/corpus_brut.csv")
print(len(corpus), "titres")
corpus.sample(10, random_state=42)

803 titres


,titre,source,url,date,rubrique_site,theme_rubrique,date_collecte
192,Le Ministère de la Santé lance une campagne d’...,La Nation,https://www.lanation.dj/le-ministere-de-la-san...,NaN,sante,sante,2026-11-19
294,03 déc.GOUVERNEMENTLe chef de l’Etat préside l...,ADI,https://www.adi.dj/article/12560,10 décembre 2024,GOUVERNEMENT,politique,2026-11-19
168,Semaine mondiale de l’allaitement : un élan co...,La Nation,https://www.lanation.dj/semaine-mondiale-de-la...,NaN,sante,sante,2026-11-19
521,08 févr.ÉCONOMIEDoraleh : la SGTD renforce sa ...,ADI,https://www.adi.dj/article/12776,23 avril 2025,ECONOMIE,economie,2026-11-19
535,14 oct.SANTÉDes médecins chinois décorés de l'...,ADI,https://www.adi.dj/article/139001,6 novembre 2026,SANTE,sante,2026-11-19
789,29 avr.CULTUREDjibouti : « La Femme du fossoye...,ADI,https://www.adi.dj/article/9735,25 février 2024,CULTURE,societe,2026-11-19
591,09 avr.SANTÉ75ème anniversaire de l'OMS : Décl...,ADI,https://www.adi.dj/article/10448,12 juin 2024,SANTE,sante,2026-11-19
323,16 maiGOUVERNEMENTLe chef de l'Etat préside le...,ADI,https://www.adi.dj/article/10546,24 Mars 2016,GOUVERNEMENT,politique,2026-11-19
218,Sport et jeunesse : Le Secrétariat d’État aux ...,La Nation,https://www.lanation.dj/sport-et-jeunesse-le-s...,NaN,sports,sport,2026-11-19
405,20 juinPARLEMENTLe président de l’Assemblée na...,ADI,https://www.adi.dj/article/11990,2 mars 2025,PARLEMENT,politique,2026-11-19


In [2]:
import re

MAJ = "A-ZÉÈÊÀÂÎÔÛÇ"
MIN = "a-zéèêàâîôûç"

AVEC_DATE = rf"^\d{{1,2}}\s+[a-zéû]+\.?\s*[{MAJ}]{{3,}}?(?=[{MAJ}][{MIN}]|\d|[«\"“'\s])"
SANS_DATE = rf"^[{MAJ}]{{4,}}?(?=[{MAJ}][{MIN}])"
CHAPEAU   = rf"[{MAJ}][{MAJ}\s'\-]{{2,}},?\s*\d{{1,2}}\s+[{MIN}A-Z]+\s+20\d\d.*$"
COLLE     = rf"(?<=[{MIN}»\)\.!?])(?=[{MAJ}][{MIN}]).*$"

def nettoyer_titre(titre):
    titre = re.sub(AVEC_DATE, "", titre)
    titre = re.sub(SANS_DATE, "", titre)
    titre = re.sub(CHAPEAU, "", titre)
    titre = re.sub(r"LIRE LA SUITE.*$", "", titre)
    titre = re.sub(COLLE, "", titre)
    return titre.strip()

corpus["titre"] = corpus["titre"].apply(nettoyer_titre)
corpus = corpus[corpus["titre"].str.len() > 10].drop_duplicates(subset="titre")
print(len(corpus), "titres après nettoyage")

suspects = corpus[corpus["titre"].str.contains(r"\(#ADI\)|LIRE LA SUITE") | (corpus["titre"].str.len() > 200)]
print(len(suspects), "titres suspects")
pd.set_option("display.max_colwidth", None)
suspects["titre"]

797 titres après nettoyage
98 titres suspects


269                                                                                                                                                                                                                                                                                                                                                                                                                            Le chef de l’Etat préside les travaux de la 17ème séance du Conseil des ministresDJIBOUTI, 18 novembre 2025 (ADI) - Sous la Présidence de M. Ismail Omar Guelleh, Président de la République, Chef du Gouvernement a eu lieu ce mardi 18 Novembre 2025, la 17ème Séance du Conseil des Ministres.\r\nOnt été examinés et approuvés les textes suivants :\r\nMINISTERE DE LA VILLE, DE L’URBANISME ET DE L’HABITAT.\r\n1ERE QUESTION : Communication sur l’Elaboration du Schéma Directeur d’Aménagement et d’Urbanisme de l’Agglomération de Djibouti.
271                                               

In [3]:
corpus = corpus.drop(suspects.index)
print(len(corpus), "titres propres")
print(corpus["theme_rubrique"].value_counts())

echantillon = (corpus.groupby("theme_rubrique")
                     .sample(n=40, random_state=42)
                     .sample(frac=1, random_state=42)
                     .reset_index(drop=True))

echantillon["id"] = echantillon.index + 1
echantillon["theme_final"] = echantillon["theme_rubrique"]
echantillon["remarque"] = ""

colonnes = ["id", "titre", "source", "url", "date", "rubrique_site", "theme_rubrique", "theme_final", "remarque"]
echantillon[colonnes].to_csv("../data/echantillon_a_verifier.csv", index=False)
print(len(echantillon), "titres enregistrés dans data/echantillon_a_verifier.csv")

699 titres propres
theme_rubrique
politique    242
economie     130
sport        130
sante        102
societe       55
education     40
Name: count, dtype: int64
240 titres enregistrés dans data/echantillon_a_verifier.csv
